In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sb

In [222]:
df_title = pd.read_csv('../data/movie_title.csv')

In [223]:
df_review = pd.read_csv('../data/movie_review.csv')

In [224]:
df_title

,item_id,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)
...,...,...
1677,1678,Mat' i syn (1997)
1678,1679,B. Monkey (1998)
1679,1680,Sliding Doors (1998)
1680,1681,You So Crazy (1994)


In [225]:
df_review

,user_id,item_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
...,...,...,...,...
99995,880,476,3,880175444
99996,716,204,5,879795543
99997,276,1090,1,874795795
99998,13,225,2,882399156


### 두개의 테이블 합치기

In [226]:
# pd.merge(df_title, df_review, on='item_id')

#### 넛차라는 영화 스트리밍 하는 스타드업 회사에 입사  
영화 1600편의 영화가 있다.   
신규서비스라서 회원이 천명밖에 없다  
영화는 1600편이 있는데, 리뷰를 남긴 사람은 백명뿐이다  
1600편의 영화중에서도 유명한 영화만 리뷰를 남겼다
DB에 테이블은 movie, review 테이블이 있다  
영화 목록이 나오도록 sql을 만들어라

In [227]:
# => left join 사용해주기
df_movie = pd.merge(df_title, df_review, on = 'item_id', how ='left' )

In [228]:
df_movie

,item_id,title,user_id,rating,timestamp
0,1,Toy Story (1995),308,4,887736532
1,1,Toy Story (1995),287,5,875334088
2,1,Toy Story (1995),148,4,877019411
3,1,Toy Story (1995),280,4,891700426
4,1,Toy Story (1995),66,3,883601324
...,...,...,...,...,...
99995,1678,Mat' i syn (1997),863,1,889289570
99996,1679,B. Monkey (1998),863,3,889289491
99997,1680,Sliding Doors (1998),863,2,889289570
99998,1681,You So Crazy (1994),896,3,887160722


In [229]:
# groupby(item_id)로 했을 때는 결과값이 사람이 보기 불편함
df_movie.groupby('title')['rating'].mean().sort_values(ascending=False)

title
Aiqing wansui (1994)                                 5.0
Entertaining Angels: The Dorothy Day Story (1996)    5.0
Santa with Muscles (1996)                            5.0
Prefontaine (1997)                                   5.0
They Made Me a Criminal (1939)                       5.0
                                                    ... 
Pharaoh's Army (1995)                                1.0
August (1996)                                        1.0
Babyfever (1994)                                     1.0
Amityville: A New Generation (1993)                  1.0
Woman in Question, The (1950)                        1.0
Name: rating, Length: 1664, dtype: float64

In [230]:
# 타이들의 별점과 별점의 갯수를 알고싶다 => 2가지 이상일때는 [agg]:집계함수 사용
df_movie.groupby('title')['rating'].agg(['mean','count']).sort_values('mean', ascending=False)

,mean,count
title,,
Aiqing wansui (1994),5.0,1
Entertaining Angels: The Dorothy Day Story (1996),5.0,1
Santa with Muscles (1996),5.0,2
Prefontaine (1997),5.0,3
They Made Me a Criminal (1939),5.0,1
...,...,...
Pharaoh's Army (1995),1.0,1
August (1996),1.0,1
Babyfever (1994),1.0,2


In [231]:
df_movie.groupby('title')['rating'].describe().sort_values('count')

,count,mean,std,min,25%,50%,75%,max
title,,,,,,,,
Á köldum klaka (Cold Fever) (1994),1.0,3.000000,NaN,3.0,3.0,3.0,3.0,3.0
Yankee Zulu (1994),1.0,1.000000,NaN,1.0,1.0,1.0,1.0,1.0
Aiqing wansui (1994),1.0,5.000000,NaN,5.0,5.0,5.0,5.0,5.0
Window to Paris (1994),1.0,4.000000,NaN,4.0,4.0,4.0,4.0,4.0
Wings of Courage (1995),1.0,4.000000,NaN,4.0,4.0,4.0,4.0,4.0
...,...,...,...,...,...,...,...,...
Liar Liar (1997),485.0,3.156701,1.098544,1.0,3.0,3.0,4.0,5.0
Return of the Jedi (1983),507.0,4.007890,0.923955,1.0,3.0,4.0,5.0,5.0
Fargo (1996),508.0,4.155512,0.975756,1.0,4.0,4.0,5.0,5.0


## 상관계수  = > 유사도를 측정(중요!)
### 추천 시스템 만들기 => 협업 필터링 기법
- 협업필터링 : 유저 기반의 협업 필터링과 아이템 기반 협업 필터링이 있다
- [ 유저기반 ] = 성향이 비슷한 유저를 찾아서 추천하는 방법
- [ 아이템 기반 ] = 아이템이 비슷한 것을 추천하는 방법

### 아이템 기반의 추천 시스템 만들기
- => 타이틀을 컬럼으로 , 유저 아이디를 행으로, 별점을 값으로 표시하기

In [232]:
df_movie.corr(numeric_only=True)

,item_id,user_id,rating,timestamp
item_id,1.000000,0.010377,-0.189124,0.041867
user_id,0.010377,1.000000,-0.009371,0.019082
rating,-0.189124,-0.009371,1.000000,-0.012003
timestamp,0.041867,0.019082,-0.012003,1.000000


In [233]:
df_movie

,item_id,title,user_id,rating,timestamp
0,1,Toy Story (1995),308,4,887736532
1,1,Toy Story (1995),287,5,875334088
2,1,Toy Story (1995),148,4,877019411
3,1,Toy Story (1995),280,4,891700426
4,1,Toy Story (1995),66,3,883601324
...,...,...,...,...,...
99995,1678,Mat' i syn (1997),863,1,889289570
99996,1679,B. Monkey (1998),863,3,889289491
99997,1680,Sliding Doors (1998),863,2,889289570
99998,1681,You So Crazy (1994),896,3,887160722


In [234]:
df = df_movie.pivot_table(columns='title', index='user_id', values='rating')

In [235]:
# Titanic 찾기
df_movie.loc[df_movie['title'].str.contains('Titanic'),]

,item_id,title,user_id,rating,timestamp
48602,313,Titanic (1997),240,5,885775604
48603,313,Titanic (1997),134,5,891732150
48604,313,Titanic (1997),319,5,889816026
48605,313,Titanic (1997),111,4,891679901
48606,313,Titanic (1997),236,4,890115777
...,...,...,...,...,...
48947,313,Titanic (1997),829,4,891204191
48948,313,Titanic (1997),818,4,891870173
48949,313,Titanic (1997),487,3,883439795
48950,313,Titanic (1997),838,5,887060659


### Titanic (1997)과 같은 영화 추천하기

In [236]:
# 상관관계 수치가 1에 가까울수록 비슷한 영화
# => 하지만 상관관계가 정확하지 않다 => 리뷰 개수가 적은 영화도 있을 수 있기 때문에
df.corr()['Titanic (1997)'].sort_values(ascending = False).head(10)

title
Simple Wish, A (1997)                      1.0
Newton Boys, The (1998)                    1.0
Underworld (1997)                          1.0
Amazing Panda Adventure, The (1995)        1.0
Vermin (1998)                              1.0
Getting Even with Dad (1994)               1.0
Crossfire (1947)                           1.0
Dangerous Ground (1997)                    1.0
Wooden Man's Bride, The (Wu Kui) (1994)    1.0
For Ever Mozart (1996)                     1.0
Name: Titanic (1997), dtype: float64

In [237]:
# 리뷰의 갯수를 정하여 정확도, 유사도를 올려보자
# corr(min_periods=입력값)
df.corr(min_periods = 50)['Titanic (1997)'].sort_values(ascending = False).head(10)

title
Titanic (1997)                       1.000000
River Wild, The (1994)               0.497600
Abyss, The (1989)                    0.472103
True Lies (1994)                     0.435104
Last of the Mohicans, The (1992)     0.427239
Die Hard (1988)                      0.403124
Air Force One (1997)                 0.383730
Jurassic Park (1993)                 0.382785
E.T. the Extra-Terrestrial (1982)    0.373201
Pretty Woman (1990)                  0.372027
Name: Titanic (1997), dtype: float64

In [238]:
df.corr(min_periods = 50)['Star Wars (1977)'].sort_values(ascending = False).head(10)

title
Star Wars (1977)                                      1.000000
Empire Strikes Back, The (1980)                       0.747981
Return of the Jedi (1983)                             0.672556
Raiders of the Lost Ark (1981)                        0.536117
Austin Powers: International Man of Mystery (1997)    0.377433
Sting, The (1973)                                     0.367538
Indiana Jones and the Last Crusade (1989)             0.350107
Pinocchio (1940)                                      0.347868
Things to Do in Denver when You're Dead (1995)        0.337714
Shadowlands (1993)                                    0.332941
Name: Star Wars (1977), dtype: float64

### 유저기반 추천 시스템 만들기  
- 개인화 추천  

In [239]:
# 예를 들어 user_id가 895인 사람

In [240]:
df_movie['user_id'].value_counts()

user_id
405    737
655    685
13     636
450    540
276    518
      ... 
36      20
364     20
685     20
631     20
418     20
Name: count, Length: 943, dtype: int64

In [241]:
my_review = df_movie.loc[df_movie['user_id'] == 895, ['user_id','title','rating'] ].reset_index(drop=True)

In [242]:
my_review.head()

,user_id,title,rating
0,895,Toy Story (1995),4
1,895,Mighty Aphrodite (1995),5
2,895,Star Wars (1977),5
3,895,Fargo (1996),4
4,895,"Rock, The (1996)",3


In [243]:
df_corr = df.corr(min_periods = 50)   # min_periods = 리뷰의 갯수 지정 = 중요!

### [ df_corr ]이 상관계수 데이터프레임을 가지고 , 유저가 본 영화의 유사도를 측정해서  
=> 추천 영화를 뽑아오되, 유저가 준 별점을 반영해서 가중치를 준다

In [244]:
title = my_review['title'][1]

In [245]:
title

'Mighty Aphrodite (1995)'

In [246]:
my_review['rating'][1]

np.int64(5)

In [247]:
df_recom = (df_corr[title].dropna().sort_values(ascending = False)*my_review['rating'][1]).to_frame()

In [248]:
df_recom.columns = ['weight']

In [249]:
df_recom.reset_index(inplace=True)

In [250]:
my_review.shape[0]

20

In [251]:
movie_list = pd.DataFrame()

In [257]:
# 1. 로그인한 유저의 영화 별점 데이터를 가져온다
my_review = df_movie.loc[df_movie['user_id'] == 895, ['user_id','title','rating'] ].reset_index(drop=True)
# 2. 비어있는 데이터프레임을 하나 만든다 => 추천 영화 목록을 담기
movie_list = pd.DataFrame()
# 3. 유저가 본 영화들의 추천영화들을 전부 가져온다
for index in range(0,my_review.shape[0]) :
    title = my_review['title'][index]
    df_recom = (df_corr[title].dropna().sort_values(ascending = False)*my_review['rating'][index]).to_frame()
    df_recom.columns = ['weight']
    df_recom.reset_index(inplace=True)
    movie_list = pd.concat( [movie_list, df_recom] )  

# 이미 본 영화는 제거 
movie_list.set_index('title', inplace=True)
for title in my_review['title'] : 
    if title in movie_list.index :
        movie_list.drop(title, inplace=True)
    
# => 중복 영화는 weight가 높은것으로 놔두고 
movie_list = movie_list.groupby('title')['weight'].max()
    
# =>최상위 10개만 가져온다
print(movie_list.sort_values(ascending = False).head(10))

title
Raiders of the Lost Ark (1981)               3.824426
Empire Strikes Back, The (1980)              3.749096
Independence Day (ID4) (1996)                3.514279
Godfather, The (1972)                        3.486659
Indiana Jones and the Last Crusade (1989)    3.446717
Silence of the Lambs, The (1991)             3.382142
Back to the Future (1985)                    3.363502
Fugitive, The (1993)                         3.287726
Pulp Fiction (1994)                          3.285433
Terminator, The (1984)                       3.283160
Name: weight, dtype: float64


In [253]:
from sklearn.metrics.pairwise import cosine_similarity

In [254]:
# 코사인 유사도에 넣을때에는 nan을 0으로 바꾼 후에 넣는다
# transpose() = 행과 열을 바꾼다
values = cosine_similarity( df.transpose().fillna(0))

In [255]:
df_corr = pd.DataFrame( values, index= df.columns, columns=df.columns)

In [185]:
df_corr.head()

title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
title,,,,,,,,,,,,,,,,,,,,,
'Til There Was You (1997),1.000000,0.000000,0.024561,0.099561,0.185236,0.159265,0.000000,0.052203,0.000000,0.033326,...,0.000000,0.000000,0.000000,0.027774,0.118840,0.142315,0.029070,0.000000,0.110208,0.000000
1-900 (1994),0.000000,1.000000,0.014139,0.009294,0.007354,0.004702,0.010055,0.067038,0.000000,0.000000,...,0.152499,0.015484,0.000000,0.069284,0.018243,0.023408,0.006694,0.079640,0.042295,0.000000
101 Dalmatians (1996),0.024561,0.014139,1.000000,0.167006,0.061105,0.143878,0.203781,0.225803,0.027642,0.092337,...,0.000000,0.021965,0.030905,0.274877,0.204267,0.101199,0.056976,0.172155,0.045714,0.000000
12 Angry Men (1957),0.099561,0.009294,0.167006,1.000000,0.056822,0.167235,0.304078,0.422506,0.072682,0.394854,...,0.060946,0.016502,0.000000,0.403270,0.259436,0.145519,0.105226,0.038901,0.060101,0.081261
187 (1997),0.185236,0.007354,0.061105,0.056822,1.000000,0.132327,0.042928,0.065060,0.043133,0.027300,...,0.000000,0.141997,0.000000,0.068257,0.067786,0.091293,0.099490,0.025184,0.142667,0.096449
